# **Проектная работа №4. Анализ динамики изменений земного покрова на последовательности разновременных спутниковых снимков с использованием спектральных индексов**

## **1. Цель работы**

Выполнить количественный анализ динамики изменения классов земного покрова на выбранной территории за период с 2019 по 2024 год на основе разновременных спутниковых снимков Sentinel-2 и провести сравнительную оценку результатов, полученных методом пороговой сегментации на основе спектральных индексов и с использованием готовых карт классификации Dynamic World.

## **2. Задачи работы**

1.  **Подготовить** последовательность разновременных снимков (снимки Sentinel-2 и карты Dynamic World) за 2019-2024 гг.;
2.  **Классифицировать** земной покров на снимках Sentinel-2, применив метод пороговой сегментации по спектральным индексам;
3.  **Рассчитать и сравнить** количественные показатели динамики (площади, темпы роста) для двух методов классификации: по индексам и по данным Dynamic World;
4.  **Оценить** выявленные тренды изменения покрова и **сформулировать** выводы о сопоставимости результатов двух методов анализа.

## **3. Загрузка данных для анализа из Google Earth Engine**

> **Примечание.** В качестве примера реализации можно использовать код из лекционных материалов, доступный по ссылке: https://u.to/nWdJIg. Для выполнения текущей работы актуальны только **Раздел 1** и **Раздел 2** указанного источника.





### **3.1. Настройка рабочей среды и аутентификация**

In [ ]:
%%capture
%pip install -q earthengine-api geemap geedim rasterio numpy pandas matplotlib

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import transform_bounds, transform_geom
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display, Markdown

YEARS = list(range(2019, 2025))
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
SCALE = 10
NODATA = -9999
ROOT = Path('5_files')
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
for folder in (DATA, RESULTS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10,
                     'axes.titlesize': 12, 'axes.titleweight': 'bold'})
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda value: f'{value:.2f}')

required_files = [DATA / f'{prefix}_summer_{year}.tif'
                  for year in YEARS
                  for prefix in ('sentinel2', 'DYNAMICWORLD')]
download_needed = not all(path.is_file() for path in required_files)

if download_needed:
    import ee
    import geemap
    project_id = input('ID проекта Google Cloud с Earth Engine: ').strip()
    if not project_id:
        raise ValueError('Для загрузки данных нужен ID проекта Earth Engine.')
    ee.Authenticate()
    # Только пользовательские учётные данные, без поиска серверной авторизации.
    credentials = ee.data.get_persistent_credentials()
    ee.Initialize(credentials=credentials, project=project_id)
    print('Подключение к Earth Engine выполнено.')
else:
    print('Используются ранее загруженные GeoTIFF из 5_files/data.')


### **3.2. Определение области интереса**

- Выберите территорию размером не менее 10×10 км (100 км²) с разнообразным земным покровом
- Обязательно должны присутствовать: водные объекты, растительность, застройка, открытая почва
- Зафиксируйте координаты углов ROI для воспроизводимости результатов

In [ ]:
# Клязьминское водохранилище, леса, поля и застройка севернее Москвы.
ROI_BOUNDS = [37.40, 55.92, 37.62, 56.04]
west, south, east, north = ROI_BOUNDS
ROI_COORDS = [[west, south], [east, south], [east, north],
              [west, north], [west, south]]
roi_geojson = {'type': 'Polygon', 'coordinates': [ROI_COORDS]}
longitude = (west + east) / 2
latitude = (south + north) / 2
utm_zone = int((longitude + 180) // 6) + 1
EXPORT_CRS = f'EPSG:{(32600 if latitude >= 0 else 32700) + utm_zone}'

# Общая сетка для всех лет и обоих методов.
left, bottom, right, top = transform_bounds(
    'EPSG:4326', EXPORT_CRS, *ROI_BOUNDS, densify_pts=21)
left = np.floor(left / SCALE) * SCALE
bottom = np.floor(bottom / SCALE) * SCALE
right = np.ceil(right / SCALE) * SCALE
top = np.ceil(top / SCALE) * SCALE
WIDTH = int(round((right - left) / SCALE))
HEIGHT = int(round((top - bottom) / SCALE))
GRID = from_origin(left, top, SCALE, SCALE)
GRID_TRANSFORM = [GRID.a, GRID.b, GRID.c, GRID.d, GRID.e, GRID.f]
roi_utm = transform_geom('EPSG:4326', EXPORT_CRS, roi_geojson)
roi_mask = geometry_mask([roi_utm], out_shape=(HEIGHT, WIDTH),
                         transform=GRID, invert=True)
roi_area_ha = roi_mask.sum() * SCALE ** 2 / 10000
extent_km = (0, WIDTH * SCALE / 1000, 0, HEIGHT * SCALE / 1000)

print('Углы ROI [долгота, широта]:', ROI_COORDS)
print('Проекция:', EXPORT_CRS)
print(f'Охват: около {(right-left)/1000:.1f} × {(top-bottom)/1000:.1f} км')
print(f'Площадь ROI по сетке: {roi_area_ha / 100:.2f} км²')
print(f'Пиксель: {SCALE} × {SCALE} м; сетка: {WIDTH} × {HEIGHT}')
roi = ee.Geometry.Polygon([ROI_COORDS], geodesic=False) if download_needed else None


Для расчётов используется общая сетка 10×10 м в UTM. Каналы B11 и B12 приводятся к ней с исходного разрешения 20 м.

### **3.3. Функции для загрузки и обработки данных Sentinel-2**

In [ ]:
def load_sentinel2_collection(roi, start_date, end_date, cloud_percentage=20):
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(roi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_percentage)))

    def mask_clouds(image):
        scl = image.select('SCL')
        # 0 и 1 — отсутствующие/испорченные данные; 3, 8, 9, 10 — тени и облака.
        mask = scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
        mask = mask.And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
        return (image.select(BANDS).updateMask(mask).divide(10000)
                .copyProperties(image, ['system:time_start', 'system:index']))

    return collection.map(mask_clouds)


def create_median_composite(collection, roi):
    return collection.median().clip(roi)


### **3.4. Создание медианных композитов за летние периоды 2019-2024**

- Используйте медиану для минимизации влияния облачности
- Фильтрация по облачности: CLOUDY_PIXEL_PERCENTAGE < 20%
- Маскирование облаков и теней по каналу SCL (см. лекцию, раздел 2.2)


In [ ]:
sentinel_composites = {}
source_info = {'roi': roi_geojson, 'crs': EXPORT_CRS, 'scale_m': SCALE,
               'bands': BANDS, 'scl_excluded': [0, 1, 3, 8, 9, 10],
               'cloud_percentage_lt': 20, 'years': {}}
if download_needed:
    for year in YEARS:
        # Правая граница filterDate не включается: так учитывается весь август.
        start, end = f'{year}-06-01', f'{year}-09-01'
        collection = load_sentinel2_collection(roi, start, end)
        image_ids = collection.aggregate_array('system:index').getInfo()
        if not image_ids:
            raise ValueError(f'Нет подходящих снимков Sentinel-2 за {year} год.')
        sentinel_composites[year] = create_median_composite(collection, roi)
        source_info['years'][str(year)] = {'sentinel2_images': image_ids,
                                          'start': start, 'end_exclusive': end}
        print(f'{year}: Sentinel-2 — {len(image_ids)} снимков')


### **3.5. Функции для загрузки и обработки данных карт классификации**

**Используйте коллекцию Dynamic World (Google/WRI)**

**Коллекция в GEE:** `ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")`
- **Период:** с июня 2015 года по настоящее время (обновляется постоянно)
- **Разрешение:** 10 метров
- **Особенности:**
  - Предоставляет вероятности для каждого класса (не жесткую классификацию)
  - Обновляется почти в реальном времени для каждого снимка Sentinel-2
  - 9 классов земного покрова


```python
# Пример загрузки Dynamic World

def load_landcover_data(roi, start_date, end_date, composite='mode'):
    """
    Загружает карты классификации Google Dynamic World (10м).
    
    Классы: 0-Вода, 1-Деревья, 2-Трава, 3-Затопл.растительность,
            4-Культуры, 5-Кустарники, 6-Застройка, 7-Голая земля, 8-Снег/лед
    
    Args:
        roi: область интереса
        start_date, end_date: период 'YYYY-MM-DD'
        composite: 'mode'|'first'|'mosaic'|'collection'
    """
    
    labels = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
        .filterBounds(roi) \
        .filterDate(start_date, end_date) \
        .select('label')
    
    if composite == 'collection':
        return labels.map(lambda img: img.clip(roi))
    
    return getattr(labels, composite)().clip(roi)

# Загрузка с композитом по моде (наиболее частый класс)
landcover = load_landcover_data(roi, '2024-06-01', '2024-08-31')

# Параметры визуализации для Dynamic World
landcover_vis = {
    'min': 0,
    'max': 8,
    'palette': [
        '419bdf',  # 0 - Вода - синий
        '397d49',  # 1 - Деревья - темно-зеленый  
        '88b053',  # 2 - Трава - светло-зеленый
        '7a87c6',  # 3 - Затопленная растительность - сине-фиолетовый
        'e49635',  # 4 - Культуры - оранжево-желтый
        'dfc35a',  # 5 - Кустарники - песочный
        'c4281b',  # 6 - Застройка - красный
        'a59b8f',  # 7 - Голая земля - серо-коричневый
        'b39fe1'   # 8 - Снег и лед - светло-фиолетовый (для контраста)
    ]
}

# Визуализация на карте
map_dw = geemap.Map()
map_dw.centerObject(roi, 11)
map_dw.addLayer(landcover, landcover_vis, 'Dynamic World')
map_dw

```

In [ ]:
DW_NAMES = ['Вода', 'Деревья', 'Трава', 'Затопленная растительность',
            'Культуры', 'Кустарники', 'Застройка', 'Голая земля', 'Снег и лед']
DW_COLORS = ['#419bdf', '#397d49', '#88b053', '#7a87c6', '#e49635',
             '#dfc35a', '#c4281b', '#a59b8f', '#b39fe1']


def load_landcover_data(roi, start_date, end_date, composite='mode'):
    labels = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
              .filterBounds(roi).filterDate(start_date, end_date).select('label'))
    if composite == 'collection':
        return labels
    if composite != 'mode':
        raise ValueError('Для категориальных меток используется мода.')
    return labels.mode().toInt16().clip(roi)


### **3.6. Создание карт классификации за летние периоды 2019-2024**

- Для карт классов используйте моду — наиболее частый класс в каждом пикселе (`composite='mode'`): медиана номеров классов не имеет смысла

In [ ]:
dw_composites = {}
if download_needed:
    for year in YEARS:
        start, end = f'{year}-06-01', f'{year}-09-01'
        labels = load_landcover_data(roi, start, end, composite='collection')
        image_ids = labels.aggregate_array('system:index').getInfo()
        if not image_ids:
            raise ValueError(f'Нет карт Dynamic World за {year} год.')
        dw_composites[year] = labels.mode().toInt16().clip(roi)
        source_info['years'][str(year)]['dynamic_world_images'] = image_ids
        print(f'{year}: Dynamic World — {len(image_ids)} карт')


### **3.7. Экспорт полученных растров**

- **Медианные композиты** Sentinel-2 (6 файлов GeoTIFF):
  - `sentinel2_summer_2019.tif` (июнь-август 2019)
  - `sentinel2_summer_2020.tif` (июнь-август 2020)
  - `sentinel2_summer_2021.tif` (июнь-август 2021)
  - `sentinel2_summer_2022.tif` (июнь-август 2022)
  - `sentinel2_summer_2023.tif` (июнь-август 2023)
  - `sentinel2_summer_2024.tif` (июнь-август 2024)

- **Карты классификации** Dynamic World (6 файлов GeoTIFF):
  - `DYNAMICWORLD_summer_2019.tif` (июнь-август 2019)
  - `DYNAMICWORLD_summer_2020.tif` (июнь-август 2020)
  - `DYNAMICWORLD_summer_2021.tif` (июнь-август 2021)
  - `DYNAMICWORLD_summer_2022.tif` (июнь-август 2022)
  - `DYNAMICWORLD_summer_2023.tif` (июнь-август 2023)
  - `DYNAMICWORLD_summer_2024.tif` (июнь-август 2024)

- Экспорт выполняйте в метрической проекции UTM функцией `export_georeferenced_raster` из лекции: размер пикселя 10×10 м
- Карты Dynamic World экспортируйте с типом `int16`: при типе `uint8` значение 0 используется как «нет данных» и совпадает с классом «Вода»
- Размер пикселя 10 м соответствует собственному разрешению каналов B2, B3, B4 и B8; каналы B11 и B12 (20 м) при экспорте пересчитываются на ту же сетку. Если экспорт идёт слишком долго, увеличьте `scale` (например, до 20 или 30 м): площади в разделе 6 рассчитываются по размеру пикселя из растра

In [ ]:
def export_georeferenced_raster(image, roi, filename, scale=SCALE,
                               dtype='float32', band_names=None):
    filename = Path(filename)
    if filename.is_file():
        print('Уже сохранён:', filename.name)
        return filename
    if scale != SCALE:
        raise ValueError('Размер пикселя должен совпадать с общей сеткой.')
    temporary = filename.with_suffix('.partial.tif')
    geemap.download_ee_image(
        image=image, filename=str(temporary), region=roi,
        crs=EXPORT_CRS, crs_transform=GRID_TRANSFORM,
        shape=(HEIGHT, WIDTH), dtype=dtype, resampling='near',
        unmask_value=NODATA, max_tile_dim=512, num_threads=4)

    # Явное значение NoData сохраняет класс 0 «Вода».
    with rasterio.open(temporary) as src:
        values = src.read(masked=True).filled(NODATA).astype(dtype)
        profile = src.profile.copy()
    profile.update(nodata=NODATA, compress='deflate', predictor=3 if dtype=='float32' else 2)
    with rasterio.open(filename, 'w', **profile) as dst:
        dst.write(values)
        for number, name in enumerate(band_names or ['label'], 1):
            dst.set_band_description(number, name)
        dst.update_tags(source='Google Earth Engine', summer='June-August')
    temporary.unlink()
    print('Сохранён:', filename.name)
    return filename


if download_needed:
    for year in YEARS:
        export_georeferenced_raster(
            sentinel_composites[year], roi, DATA / f'sentinel2_summer_{year}.tif',
            dtype='float32', band_names=BANDS)
        export_georeferenced_raster(
            dw_composites[year], roi, DATA / f'DYNAMICWORLD_summer_{year}.tif',
            dtype='int16')
    (DATA / 'source_info.json').write_text(
        json.dumps(source_info, ensure_ascii=False, indent=2), encoding='utf-8')

print('Исходные GeoTIFF:', len(list(DATA.glob('*_summer_*.tif'))), 'из 12')


## **4. Работа с загруженными растрами в RasterIO**

### **4.1. Загрузка и проверка растров**

In [ ]:
def check_grid(src):
    if src.crs != rasterio.crs.CRS.from_string(EXPORT_CRS):
        raise ValueError(f'Другая проекция: {src.name}')
    if src.shape != (HEIGHT, WIDTH) or not src.transform.almost_equals(GRID):
        raise ValueError(f'Растр не совпадает с общей сеткой: {src.name}')
    if not src.crs.is_projected or src.crs.linear_units != 'metre':
        raise ValueError('Для площадей нужна метрическая система координат.')


sentinel_data = {}
metadata_rows = []
for year in YEARS:
    path = DATA / f'sentinel2_summer_{year}.tif'
    if not path.is_file():
        raise FileNotFoundError(f'Отсутствует {path}. Сначала выполняется раздел 3.')
    with rasterio.open(path) as src:
        check_grid(src)
        if src.count != len(BANDS) or list(src.descriptions) != BANDS:
            raise ValueError(f'Неверный набор или порядок каналов: {path.name}')
        sentinel_data[year] = src.read(masked=True).astype('float32').filled(np.nan)
        metadata_rows.append({'Год': year, 'CRS': str(src.crs),
                              'Размер': f'{src.width} × {src.height}',
                              'Пиксель, м': f'{src.res[0]:g} × {src.res[1]:g}',
                              'Каналы': src.count, 'NoData': src.nodata})
        raster_profile = src.profile.copy()
        pixel_area_m2 = abs(src.transform.a * src.transform.e -
                            src.transform.b * src.transform.d)
pixel_ha = pixel_area_m2 / 10000
display(pd.DataFrame(metadata_rows).set_index('Год'))

fig, axes = plt.subplots(3, 2, figsize=(11, 14), layout='constrained')
for ax, year in zip(axes.flat, YEARS):
    rgb = np.moveaxis(sentinel_data[year][[2, 1, 0]], 0, -1)
    rgb = np.clip(np.nan_to_num(rgb) / 0.3, 0, 1) ** (1 / 1.4)
    rgb[~roi_mask] = 1
    ax.imshow(rgb, extent=extent_km)
    ax.set_title(f'Sentinel-2, лето {year}')
    ax.set_xlabel('От западной границы, км')
    ax.set_ylabel('От южной границы, км')
fig.savefig(FIGURES / 'sentinel2_rgb.png', dpi=120)
plt.show()


In [ ]:
dw_data = {}
for year in YEARS:
    with rasterio.open(DATA / f'DYNAMICWORLD_summer_{year}.tif') as src:
        check_grid(src)
        labels = src.read(1, masked=True).astype('int16').filled(-1)
        invalid = (labels != -1) & ~np.isin(labels, np.arange(9))
        if np.any(invalid):
            raise ValueError(f'Неизвестная метка Dynamic World, {year}.')
        labels[~roi_mask] = -1
        dw_data[year] = labels


def plot_class_maps(maps, names, colors, title, filename):
    cmap = ListedColormap(colors)
    cmap.set_bad('#eeeeee')
    norm = BoundaryNorm(np.arange(len(names) + 1) - 0.5, len(names))
    fig, axes = plt.subplots(3, 2, figsize=(11, 15), layout='constrained')
    for ax, year in zip(axes.flat, YEARS):
        ax.imshow(np.ma.masked_less(maps[year], 0), cmap=cmap, norm=norm,
                  interpolation='nearest', extent=extent_km)
        ax.set_title(f'{title}, {year}')
        ax.set_xlabel('От западной границы, км')
        ax.set_ylabel('От южной границы, км')
    handles = [Patch(facecolor=color, edgecolor='#555555', label=name)
               for name, color in zip(names, colors)]
    handles.append(Patch(facecolor='#eeeeee', label='Нет данных / вне ROI'))
    fig.legend(handles=handles, loc='outside lower center', ncol=3, fontsize=9)
    fig.savefig(FIGURES / filename, dpi=120, bbox_inches='tight')
    plt.show()


plot_class_maps(dw_data, DW_NAMES, DW_COLORS, 'Dynamic World', 'dynamic_world.png')


### **4.2. Рассчитать все 8 индексов для каждого года (2019-2024)**


> **Примечание.** В качестве примера рассчета индексов можно использовать код из лекционных материалов, доступный по ссылке: https://u.to/4qBeIg





**Спектральные индексы** — это математические комбинации значений отражения в разных спектральных каналах, позволяющие выделить определенные типы объектов.








#### **Таблица 1. Основные спектральные индексы для анализа земного покрова**




| **Индекс** | **Формула** | **Назначение** | **Каналы Sentinel-2** |
|------------|-------------|----------------|------------------------|
| **NDVI** (Normalized Difference Vegetation Index) | (NIR - RED) / (NIR + RED) | Оценка состояния растительности | (B8 - B4) / (B8 + B4) |
| **NDWI** (Normalized Difference Water Index) | (GREEN - NIR) / (GREEN + NIR) | Выделение водных объектов | (B3 - B8) / (B3 + B8) |
| **NDBI** (Normalized Difference Built-up Index) | (SWIR - NIR) / (SWIR + NIR) | Выделение застроенных территорий | (B11 - B8) / (B11 + B8) |
| **SAVI** (Soil Adjusted Vegetation Index) | ((NIR - RED) / (NIR + RED + L)) * (1 + L), L=0.5 | Растительность с учетом почвы | ((B8 - B4) / (B8 + B4 + 0.5)) * 1.5 |
| **EVI** (Enhanced Vegetation Index) | 2.5 * ((NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)) | Улучшенный вегетационный индекс | 2.5 * ((B8 - B4) / (B8 + 6*B4 - 7.5*B2 + 1)) |
| **MNDWI** (Modified NDWI) | (GREEN - SWIR) / (GREEN + SWIR) | Модифицированный водный индекс | (B3 - B11) / (B3 + B11) |
| **BSI** (Bare Soil Index) | ((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE)) | Открытая почва | ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2)) |
| **NDSI** (Normalized Difference Snow Index) | (GREEN - SWIR) / (GREEN + SWIR) | Снежный покров | (B3 - B11) / (B3 + B11) |

NDSI и MNDWI вычисляются по одной формуле, поэтому высокие значения дают и снег, и вода. Снег отличается от воды высоким отражением в ближнем ИК-канале (B8 > 0.11), а вода ИК-излучение поглощает.


#### **Таблица 2. Интерпретация значений NDVI**



| **Диапазон NDVI** | **Тип покрова** | **Характеристика** |
|-------------------|-----------------|---------------------|
| < 0.0 | Вода | Водные поверхности, снег |
| 0.0 - 0.1 | Открытая почва | Песок, камни, бетон |
| 0.1 - 0.2 | Разреженная растительность | Степи, пустыни |
| 0.2 - 0.4 | Умеренная растительность | Кустарники, луга |
| 0.4 - 0.6 | Плотная растительность | Лиственные леса |
| > 0.6 | Очень плотная растительность | Тропические леса, хвойные леса |

#### **Таблица 3. Интерпретация значений NDWI**



| **Диапазон NDWI** | **Тип объекта** | **Описание** |
|-------------------|-----------------|--------------|
| > 0.3 | Водные объекты | Реки, озера, пруды |
| 0.0 - 0.3 | Влажная почва/растительность | Болота, заливные луга |
| -0.3 - 0.0 | Сухая растительность | Обычная растительность |
| < -0.3 | Застройка/почва | Городская застройка, открытая почва |

#### **Таблица 4. Интерпретация значений NDBI**



| **Диапазон NDBI** | **Тип территории** | **Характеристика** |
|-------------------|-------------------|---------------------|
| > 0.0 | Застроенные территории | Города, промзоны |
| -0.1 - 0.0 | Смешанные территории | Пригороды, села |
| < -0.1 | Природные территории | Леса, поля, водоемы |

#### **Таблица 5. Интерпретация значений SAVI**



| **Диапазон SAVI** | **Тип покрова** | **Описание** |
|-------------------|-----------------|---------------|
| < 0.1 | Открытая почва/вода | Водные объекты, голая почва |
| 0.1 - 0.2 | Очень разреженная растительность | Пустыни, каменистые участки |
| 0.2 - 0.4 | Разреженная растительность | Степи, саванны |
| 0.4 - 0.6 | Умеренная растительность | Сельхозугодья, луга |
| > 0.6 | Плотная растительность | Леса, густые кустарники |

#### **Таблица 6. Интерпретация значений EVI**



| **Диапазон EVI** | **Состояние растительности** | **Характеристика** |
|------------------|------------------------------|---------------------|
| < 0.0 | Нет растительности | Вода, снег, облака |
| 0.0 - 0.2 | Минимальная растительность | Голая почва, урбанизированные территории |
| 0.2 - 0.3 | Низкая биомасса | Травянистая растительность |
| 0.3 - 0.5 | Средняя биомасса | Кустарники, молодые посевы |
| 0.5 - 0.8 | Высокая биомасса | Зрелые леса, интенсивное сельское хозяйство |
| > 0.8 | Очень высокая биомасса | Тропические леса |

#### **Таблица 7. Интерпретация значений MNDWI**



| **Диапазон MNDWI** | **Тип объекта** | **Характеристика** |
|--------------------|-----------------|---------------------|
| > 0.5 | Глубокая вода | Озера, моря, водохранилища |
| 0.2 - 0.5 | Мелкая вода | Реки, каналы, мелководья |
| 0.0 - 0.2 | Влажные территории | Болота, рисовые поля |
| < 0.0 | Суша | Растительность, почва, застройка |

#### **Таблица 8. Интерпретация значений BSI**



| **Диапазон BSI** | **Тип поверхности** | **Описание** |
|-------------------|-------------------|---------------|
| > 0.1 | Открытая почва | Пашни, карьеры, пустыни |
| 0.0 - 0.1 | Частично покрытая почва | Разреженная растительность |
| -0.1 - 0.0 | Смешанный покров | Переходные зоны |
| < -0.1 | Растительность/вода | Леса, водоемы, плотная растительность |

#### **Таблица 9. Интерпретация значений NDSI**



| **Диапазон NDSI** | **Тип покрова** | **Характеристика** |
|-------------------|-----------------|---------------------|
| > 0.4 | Снежный покров | Чистый снег, ледники |
| 0.2 - 0.4 | Частичный снежный покров | Смешанный снег с растительностью |
| 0.0 - 0.2 | Возможный снег/лед | Тающий снег, иней, облака |
| -0.2 - 0.0 | Переходная зона | Влажная почва, туман |
| < -0.2 | Бесснежные территории | Растительность, почва, вода |

In [ ]:
def safe_divide(numerator, denominator):
    numerator, denominator = np.broadcast_arrays(
        np.asarray(numerator, dtype='float32'),
        np.asarray(denominator, dtype='float32'))
    valid = np.isfinite(numerator) & np.isfinite(denominator) & (np.abs(denominator) > 1e-6)
    result = np.full(numerator.shape, np.nan, dtype='float32')
    np.divide(numerator, denominator, out=result, where=valid)
    return result


def calculate_indices(data):
    blue, green, red, nir, swir1, swir2 = data
    mndwi = safe_divide(green - swir1, green + swir1)
    return {
        'NDVI': safe_divide(nir - red, nir + red),
        'NDWI': safe_divide(green - nir, green + nir),
        'NDBI': safe_divide(swir1 - nir, swir1 + nir),
        'SAVI': 1.5 * safe_divide(nir - red, nir + red + 0.5),
        'EVI': 2.5 * safe_divide(nir - red, nir + 6*red - 7.5*blue + 1),
        'MNDWI': mndwi,
        'BSI': safe_divide(swir1 + red - nir - blue, swir1 + red + nir + blue),
        'NDSI': mndwi.copy(),
    }


indices = {year: calculate_indices(sentinel_data[year]) for year in YEARS}
for year in YEARS:
    for name, values in indices[year].items():
        values[~roi_mask] = np.nan
print(f'Рассчитано {len(YEARS)} × {len(indices[2024])} = 48 индексов.')


### **4.3. Визуализировать каждый индекс для 2024 года с соответствующей цветовой шкалой**



In [ ]:
index_styles = {
    'NDVI': ('RdYlGn', -1, 1), 'NDWI': ('BrBG', -1, 1),
    'NDBI': ('RdBu_r', -1, 1), 'SAVI': ('RdYlGn', -0.5, 1),
    'EVI': ('YlGn', -0.2, 1), 'MNDWI': ('BrBG', -1, 1),
    'BSI': ('BrBG_r', -1, 1), 'NDSI': ('Blues', -1, 1),
}
fig, axes = plt.subplots(4, 2, figsize=(11, 18), layout='constrained')
for ax, (name, (cmap, vmin, vmax)) in zip(axes.flat, index_styles.items()):
    im = ax.imshow(indices[2024][name], cmap=cmap, vmin=vmin, vmax=vmax,
                   extent=extent_km)
    ax.set_title(f'{name}, 2024')
    ax.set_xlabel('От западной границы, км')
    ax.set_ylabel('От южной границы, км')
    fig.colorbar(im, ax=ax, label=name, shrink=0.75, extend='both')
fig.savefig(FIGURES / 'indices_2024.png', dpi=120)
plt.show()
print('Цветовые шкалы ограничены для отображения; исходные индексы не обрезаются.')


### **4.4. Сохранить все рассчитанные индексы в отдельные GeoTIFF файлы с сохранением геопривязки (сохранить в соответствующие папки по годам)**



In [ ]:
def save_raster(filename, values, description):
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    values = np.asarray(values)
    is_float = np.issubdtype(values.dtype, np.floating)
    dtype = 'float32' if is_float else 'int16'
    nodata = NODATA if is_float else -1
    data = np.where(np.isfinite(values), values, nodata).astype(dtype)
    profile = raster_profile.copy()
    profile.update(count=1, dtype=dtype, nodata=nodata,
                   compress='deflate', predictor=3 if is_float else 2)
    with rasterio.open(filename, 'w', **profile) as dst:
        dst.write(data, 1)
        dst.set_band_description(1, description)


for year in YEARS:
    for name, values in indices[year].items():
        save_raster(RESULTS / str(year) / f'{name}_{year}.tif', values, name)
print('48 GeoTIFF сохранены в 5_files/results по годам.')


## **5. Сегментация и классификация**

**Сегментация** — процесс разделения изображения на однородные регионы (сегменты) на основе заданных критериев .


### **5.1. Многопороговая сегментация с использованием спектральных индексов**



**Задание:** Реализовать классификацию земного покрова путем замены значений пикселей в растрах индексов на метки классов согласно пороговым правилам.

**Пояснение:** По сути необходимо преобразовать непрерывные значения индексов в дискретные классы (метки) на основе пороговых значений из Таблиц 2-9. Каждому пикселю присваивается числовая метка класса (0-7) в зависимости от значений спектральных индексов.






**Алгоритм многопороговой сегментации:**
1. Создать пустой растр для меток классов (заполнить значением -1 или 255 для "неклассифицировано")
2. Последовательно применить правила классификации
3. Каждое правило проверяется только для еще не классифицированных пикселей
4. Присвоить соответствующую метку класса пикселям, удовлетворяющим условию

**Иерархия правил классификации (применять строго последовательно):**
1. **Снег/лед (метка=1)**: (NDSI > 0.4) И (B8 > 0.11)
2. **Вода (метка=0)**: ((NDWI > 0.3) ИЛИ (MNDWI > 0.2)) И (еще не классифицировано)
3. **Застройка (метка=2)**: (NDBI > 0.0) И (NDVI < 0.2) И (еще не классифицировано)
4. **Густая растительность (метка=3)**: ((NDVI > 0.6) ИЛИ (EVI > 0.5)) И (еще не классифицировано)
5. **Умеренная растительность (метка=4)**: (0.3 ≤ NDVI ≤ 0.6) И (еще не классифицировано)
6. **Разреженная растительность (метка=5)**: (0.1 ≤ NDVI < 0.3) И (еще не классифицировано)
7. **Открытая почва (метка=6)**: (BSI > 0.0) И (NDVI < 0.1) И (еще не классифицировано)
8. **Прочее (метка=7)**: все оставшиеся неклассифицированные пиксели

**Обратите внимание:** открытая почва и пашня, как и застройка, дают NDBI > 0, поэтому правило 3 относит часть открытой почвы к застройке. Учитывайте это при интерпретации результатов и сравнении с Dynamic World.


**(МОЖЕТЕ СФОРМУЛИРОВАТЬ СВОИ ПРАВИЛА, ОСНОВЫВАЯСЬ НА ТАБЛИЦАХ, ПРИВЕДЕННЫХ ВЫШЕ!)**

In [ ]:
CLASS_NAMES = ['Вода', 'Снег/лед', 'Застройка', 'Густая растительность',
               'Умеренная растительность', 'Разреженная растительность',
               'Открытая почва', 'Прочее']
CLASS_COLORS = ['#419bdf', '#ffffff', '#c4281b', '#397d49',
                '#88b053', '#c6e48b', '#a59b8f', '#808080']


def segment_landcover(index, nir, roi_mask):
    valid = roi_mask & np.isfinite(nir)
    for values in index.values():
        valid &= np.isfinite(values)
    labels = np.full(nir.shape, -1, dtype='int16')
    ndvi = index['NDVI']
    # Порядок правил совпадает с заданием; уже присвоенные метки не меняются.
    rules = [
        (1, (index['NDSI'] > 0.4) & (nir > 0.11)),
        (0, (index['NDWI'] > 0.3) | (index['MNDWI'] > 0.2)),
        (2, (index['NDBI'] > 0) & (ndvi < 0.2)),
        (3, (ndvi > 0.6) | (index['EVI'] > 0.5)),
        (4, (ndvi >= 0.3) & (ndvi <= 0.6)),
        (5, (ndvi >= 0.1) & (ndvi < 0.3)),
        (6, (index['BSI'] > 0) & (ndvi < 0.1)),
    ]
    for code, condition in rules:
        labels[valid & (labels == -1) & condition] = code
    labels[valid & (labels == -1)] = 7
    return labels


labels_2024 = segment_landcover(indices[2024], sentinel_data[2024][3], roi_mask)
print('Метки 2024 года:', np.unique(labels_2024[labels_2024 >= 0]))



### **5.2. Подсчет количества пикселей каждого класса**



**Задание:** После замены значений индексов на метки классов, подсчитать количество пикселей для каждой метки класса.

**Пояснение:** Используйте функции numpy.unique() или аналогичные для подсчета количества пикселей с каждой меткой. Это позволит получить статистику распределения классов на изображении.

**Что нужно получить:**
- Массив уникальных меток классов (0-7)
- Количество пикселей для каждой метки
- Процентное соотношение классов от общего числа пикселей

In [ ]:
codes, counts = np.unique(labels_2024[labels_2024 >= 0], return_counts=True)
pixel_table_2024 = pd.DataFrame({'Класс': [CLASS_NAMES[c] for c in codes],
                                'Пиксели': counts,
                                'Доля ROI, %': counts / roi_mask.sum() * 100})
display(pixel_table_2024.set_index('Класс').round(2))


### **5.3. Применение сегментации ко всем временным срезам**




**Задание:** Применить разработанный алгоритм сегментации ко всем медианным композитам (2019-2024).

**Пояснение:** Использовать одинаковые пороговые значения для всех лет для обеспечения сопоставимости результатов при анализе динамики. Сохранить результаты сегментации как растры с метками классов.

**Выходные данные:**
- 6 растров с метками классов (по одному на каждый год)
- Таблица с количеством пикселей каждого класса для каждого года
- Формат сохранения: GeoTIFF с сохранением геопривязки

In [ ]:
segmented = {}
for year in YEARS:
    segmented[year] = segment_landcover(indices[year], sentinel_data[year][3], roi_mask)
    save_raster(RESULTS / str(year) / f'segmentation_{year}.tif',
                segmented[year], 'landcover_class')

# Одинаковая территория для всех лет и обоих методов.
common_mask = roi_mask.copy()
coverage = []
for year in YEARS:
    seg_valid = segmented[year] >= 0
    dw_valid = dw_data[year] >= 0
    common_mask &= seg_valid & dw_valid
    coverage.append({'Год': year, 'Sentinel-2, % ROI': 100 * (seg_valid & roi_mask).sum() / roi_mask.sum(),
                     'Dynamic World, % ROI': 100 * (dw_valid & roi_mask).sum() / roi_mask.sum()})
if not np.any(common_mask):
    raise ValueError('Нет общей области достоверных данных для всех лет.')
common_area_ha = common_mask.sum() * pixel_ha
display(pd.DataFrame(coverage).set_index('Год').round(2))
print(f'Общая область анализа: {common_area_ha / 100:.2f} км², '
      f'{100 * common_mask.sum() / roi_mask.sum():.2f}% ROI.')


def count_pixels(maps, names, mask):
    result = {}
    for year in YEARS:
        labels = maps[year][mask]
        result[year] = np.bincount(labels, minlength=len(names))
    return pd.DataFrame(result, index=pd.Index(names, name='Класс')).astype('int64')


pixel_counts = count_pixels(segmented, CLASS_NAMES, common_mask)
display(pixel_counts)
pixel_counts.to_csv(RESULTS / 'segmentation_pixels.csv', encoding='utf-8-sig')
save_raster(RESULTS / 'common_mask.tif', np.where(roi_mask, common_mask.astype('int16'), -1),
            'common_valid_area')


Площади и сравнение методов рассчитаны на пересечении действительных данных всех лет. Проценты ROI используют полную площадь выбранной территории; доля общей маски указана отдельно.

### **5.4. Визуализация результатов сегментации**






**Задание:** Создать тематические карты классификации с легендой.

**Пояснение:** Назначить каждой метке класса соответствующий цвет для визуализации. Использовать стандартную цветовую схему для классов земного покрова.

**Цветовая схема для визуализации:**
- 0 - Вода: синий (#419bdf)
- 1 - Снег/лед: белый (#ffffff)  
- 2 - Застройка: красный (#c4281b)
- 3 - Густая растительность: темно-зеленый (#397d49)
- 4 - Умеренная растительность: зеленый (#88b053)
- 5 - Разреженная растительность: светло-зеленый (#c6e48b)
- 6 - Открытая почва: коричневый (#a59b8f)
- 7 - Прочее: серый (#808080)

In [ ]:
plot_class_maps(segmented, CLASS_NAMES, CLASS_COLORS,
                'Сегментация по индексам', 'segmentation.png')


## **6. Анализ динамики изменений на сегментированных растрах**

### **6.1. Рассчитать площади всех классов для каждого года**



**Формула перевода пикселей в площадь:**
```
Площадь (га) = Количество_пикселей × (Разрешение_м)² / 10000
Площадь (км²) = Количество_пикселей × (Разрешение_м)² / 1000000
```
Для Sentinel-2: разрешение = 10 м, следовательно:
- 1 пиксель = 100 м² = 0.01 га
- 100 пикселей = 1 га
- 10000 пикселей = 1 км²

Формула верна для растров в метрической проекции (UTM), где размер пикселя задан в метрах; размер пикселя берите из растра (`src.res`).

In [ ]:
areas_ha = pixel_counts * pixel_ha
areas_km2 = areas_ha / 100
percent_roi = pixel_counts / roi_mask.sum() * 100
for title, table, filename in [
        ('Площадь, га', areas_ha, 'segmentation_area_ha.csv'),
        ('Площадь, км²', areas_km2, 'segmentation_area_km2.csv'),
        ('Доля общей площади ROI, %', percent_roi, 'segmentation_percent_roi.csv')]:
    print(title)
    display(table.round(2))
    table.to_csv(RESULTS / filename, encoding='utf-8-sig')
print('Временные ряды рассчитаны на общей маске; пропуски не считаются классом «Прочее».')


### **6.2. Построить графики динамики изменений для каждого класса**



In [ ]:
def plot_dynamics(areas, names, colors, title, filename, comparison=None):
    rows = int(np.ceil(len(names) / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(12, rows * 3),
                             layout='constrained', squeeze=False)
    for ax, name, color in zip(axes.flat, names, colors):
        line_color = '#777777' if color.lower() == '#ffffff' else color
        ax.plot(YEARS, areas.loc[name, YEARS], 'o-', color=line_color,
                 label='По индексам' if comparison is not None else None)
        if comparison is not None:
            ax.plot(YEARS, comparison.loc[name, YEARS], 's--',
                     color='#202020', label='Dynamic World')
            ax.legend(fontsize=8)
        ax.set_title(name)
        ax.set_xlabel('Год')
        ax.set_ylabel('Площадь, га')
        ax.set_xticks(YEARS)
        ax.grid(alpha=0.25)
    for ax in list(axes.flat)[len(names):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    fig.savefig(FIGURES / filename, dpi=120)
    plt.show()


plot_dynamics(areas_ha, CLASS_NAMES, CLASS_COLORS,
              'Динамика площадей по спектральным индексам', 'segmentation_dynamics.png')


### **6.3. Рассчитать все статистические показатели**



**Обязательные метрики для каждого класса:**

1. **Абсолютные показатели:**
   - Площадь в га и км² для каждого года
   - Количество пикселей
   - Процент от общей площади ROI

2. **Показатели изменений:**
   - Абсолютное изменение площади (га): ΔS = S₂₀₂₄ - S₂₀₁₉
   - Относительное изменение (%): (S₂₀₂₄ - S₂₀₁₉) / S₂₀₁₉ × 100%
   - Среднегодовой прирост (га/год): ΔS / количество_лет (2024 - 2019 = 5)
   - Среднегодовой темп роста (%/год): ((S₂₀₂₄/S₂₀₁₉)^(1/5) - 1) × 100%

3. **Статистика временного ряда:**
   - Коэффициент вариации: CV = σ/μ × 100% (вспомните материал прошлого семестра, а именно работу в numpy)
   - Стандартное отклонение площадей
   - Наклон линейного тренда площади (га/год) и коэффициент детерминации R² (`np.polyfit`)

In [ ]:
def calculate_statistics(areas):
    values = areas.loc[:, YEARS].to_numpy(dtype=float)
    first, last = values[:, 0], values[:, -1]
    delta = last - first
    relative = np.full(len(first), np.nan)
    cagr = np.full(len(first), np.nan)
    np.divide(delta * 100, first, out=relative, where=first > 0)
    cagr[first > 0] = ((last[first > 0] / first[first > 0]) **
                       (1 / (YEARS[-1] - YEARS[0])) - 1) * 100
    mean = values.mean(axis=1)
    std = values.std(axis=1, ddof=0)
    cv = np.full(len(first), np.nan)
    np.divide(std * 100, mean, out=cv, where=mean > 0)
    x = np.asarray(YEARS, dtype=float) - YEARS[0]
    slopes, r_squared = [], []
    for row in values:
        slope, intercept = np.polyfit(x, row, 1)
        prediction = slope * x + intercept
        total = np.sum((row - row.mean()) ** 2)
        r2 = 1 - np.sum((row - prediction) ** 2) / total if total > 1e-12 else 0.0
        slopes.append(0.0 if np.ptp(row) < 1e-10 else slope)
        r_squared.append(float(np.clip(r2, 0, 1)))
    stats = pd.DataFrame({
        'S_2019, га': first, 'S_2024, га': last,
        'ΔS, га': delta, 'Δ, %': relative,
        'Прирост, га/год': delta / (YEARS[-1] - YEARS[0]),
        'Темп, %/год': cagr, 'Среднее, га': mean,
        'σ, га': std, 'CV, %': cv,
        'Наклон, га/год': slopes, 'R²': r_squared,
    }, index=areas.index)
    stats['Значимое изменение'] = ((stats['Δ, %'].abs() > 10) &
                                  (stats['ΔS, га'].abs() > 10) & (stats['R²'] >= 0.5))
    return stats


statistics = calculate_statistics(areas_ha)
display(statistics.round(2))
statistics.to_csv(RESULTS / 'segmentation_statistics.csv', encoding='utf-8-sig')
print('σ рассчитано по всем шести годам (ddof=0). При нулевой начальной площади Δ% и темп не определены.')


### **6.4. Определить какие классы показали наибольшие изменения**



**Критерии определения значимых изменений:**
- Изменение площади > 10% от начального значения
- Абсолютное изменение > 10 га
- Изменение направленное: R² линейного тренда ≥ 0.5 (иначе это межгодовые колебания)
- Ранжировать классы по величине относительных изменений

In [ ]:
def ranked_changes(stats):
    ranking = stats.assign(_rank=stats['Δ, %'].abs()).sort_values('_rank', ascending=False)
    significant = ranking[ranking['Значимое изменение']]
    columns = ['S_2019, га', 'S_2024, га', 'ΔS, га', 'Δ, %', 'R²']
    print('Ранжирование по модулю относительного изменения:')
    display(ranking[columns].round(2))
    print('Направленные изменения по трём критериям задания:')
    if significant.empty:
        print('Классов, одновременно отвечающих всем трём критериям, нет.')
    else:
        display(significant[columns].round(2))
    new_classes = stats[(stats['S_2019, га'] == 0) & (stats['S_2024, га'] > 0)]
    if not new_classes.empty:
        print('Появились при нулевой начальной площади:', ', '.join(new_classes.index),
              '(относительное изменение не определено).')
    return significant


significant_seg = ranked_changes(statistics)


### **6.5. Создать сводную таблицу результатов**



| Класс | S_2019 (га) | S_2024 (га) | ΔS (га) | Δ% | Темп (%/год) | Тренд | CV (%) | R² |
|-------|-------------|-------------|---------|-----|--------------|-------|-----|-----|
| Вода | ... | ... | ... | ... | ... | ↑/↓/→ | ... | ... |
| Растительность | ... | ... | ... | ... | ... | ↑/↓/→ | ... | ... |
| ... | ... | ... | ... | ... | ... | ... | ... | ... |

In [ ]:
def trend_name(row):
    if row['Среднее, га'] == 0:
        return 'нет класса'
    if row['Значимое изменение']:
        return '↑' if row['Наклон, га/год'] > 0 else '↓'
    if row['CV, %'] > 10:
        return 'колебания'
    return '→'


def summary_table(stats):
    table = stats[['S_2019, га', 'S_2024, га', 'ΔS, га', 'Δ, %', 'Темп, %/год',
                   'CV, %', 'R²']].copy()
    table.insert(5, 'Тренд', stats.apply(trend_name, axis=1))
    return table


summary_seg = summary_table(statistics)
display(summary_seg.round(2))
summary_seg.to_csv(RESULTS / 'segmentation_summary.csv', encoding='utf-8-sig')
print('↑/↓ — направленное изменение по критериям задания; → — малые колебания; «колебания» — без устойчивого тренда.')


### **6.6. Карта изменений и матрица переходов**

**Задание:** Постройте карту пикселей, сменивших класс с 2019 по 2024 год, и матрицу переходов — площадь (га), перешедшую из каждого класса в каждый. Определите, где сосредоточены основные изменения и какие переходы преобладают.

In [ ]:
def transition_matrix(first, last, mask, names):
    size = len(names)
    pairs = first[mask].astype('int64') * size + last[mask]
    matrix = np.bincount(pairs, minlength=size*size).reshape(size, size) * pixel_ha
    return pd.DataFrame(matrix, index=names, columns=names)


transitions = transition_matrix(segmented[2019], segmented[2024], common_mask, CLASS_NAMES)
transitions.index.name = '2019 → 2024'
display(transitions.round(2))
transitions.to_csv(RESULTS / 'transitions_2019_2024_ha.csv', encoding='utf-8-sig')
changed = common_mask & (segmented[2019] != segmented[2024])
change_map = np.full(common_mask.shape, -1, dtype='int16')
change_map[common_mask] = changed[common_mask].astype('int16')
save_raster(RESULTS / 'changes_2019_2024.tif', change_map, 'changed_class')

fig, ax = plt.subplots(figsize=(8, 7), layout='constrained')
change_cmap = ListedColormap(['#e1e6eb', '#c7363b'])
change_cmap.set_bad('#ffffff')
ax.imshow(np.ma.masked_less(change_map, 0), cmap=change_cmap, vmin=0, vmax=1,
           interpolation='nearest', extent=extent_km)
ax.set_title('Изменения классов: 2019 → 2024')
ax.set_xlabel('От западной границы, км')
ax.set_ylabel('От южной границы, км')
ax.legend(handles=[Patch(facecolor='#e1e6eb', label='Класс сохранился'),
                    Patch(facecolor='#c7363b', label='Класс изменился')], loc='upper left')
fig.savefig(FIGURES / 'changes_2019_2024.png', dpi=120)
plt.show()

transition_rows = []
for i, source in enumerate(CLASS_NAMES):
    for j, target in enumerate(CLASS_NAMES):
        if i != j and transitions.iloc[i, j] > 0:
            transition_rows.append({'Из класса': source, 'В класс': target,
                                    'Площадь, га': transitions.iloc[i, j]})
top_transitions = pd.DataFrame(transition_rows, columns=['Из класса', 'В класс', 'Площадь, га'])
top_transitions = top_transitions.sort_values('Площадь, га', ascending=False).reset_index(drop=True)
print('Основные переходы:')
display(top_transitions.head(5).round(2))

# Строки растра идут с севера на юг.
quadrants = {'Северо-запад': (slice(0, HEIGHT//2), slice(0, WIDTH//2)),
             'Северо-восток': (slice(0, HEIGHT//2), slice(WIDTH//2, WIDTH)),
             'Юго-запад': (slice(HEIGHT//2, HEIGHT), slice(0, WIDTH//2)),
             'Юго-восток': (slice(HEIGHT//2, HEIGHT), slice(WIDTH//2, WIDTH))}
spatial_rows = []
for name, window in quadrants.items():
    valid_count = common_mask[window].sum()
    count = changed[window].sum()
    spatial_rows.append({'Часть ROI': name, 'Изменилось, га': count * pixel_ha,
                         'Доля изменений, %': 100*count/valid_count if valid_count else np.nan})
spatial_changes = pd.DataFrame(spatial_rows).set_index('Часть ROI')
display(spatial_changes.round(2))
changed_area_ha = changed.sum() * pixel_ha
print(f'Класс изменился на {changed_area_ha:.2f} га '
      f'({100*changed.sum()/common_mask.sum():.2f}% общей области анализа).')


## **7. Сравнение с картами Dynamic World**

### **7.1. Рассчитать площади всех классов для каждого года по картам классификации**



In [ ]:
dw_pixels = count_pixels(dw_data, DW_NAMES, common_mask)
dw_areas_ha = dw_pixels * pixel_ha
dw_areas_km2 = dw_areas_ha / 100
dw_percent_roi = dw_pixels / roi_mask.sum() * 100
for title, table, filename in [
        ('Dynamic World: количество пикселей', dw_pixels, 'dynamic_world_pixels.csv'),
        ('Dynamic World: площадь, га', dw_areas_ha, 'dynamic_world_area_ha.csv'),
        ('Dynamic World: площадь, км²', dw_areas_km2, 'dynamic_world_area_km2.csv'),
        ('Dynamic World: доля ROI, %', dw_percent_roi, 'dynamic_world_percent_roi.csv')]:
    print(title)
    display(table.round(2))
    table.to_csv(RESULTS / filename, encoding='utf-8-sig')


### **7.2. Построить графики динамики изменений для каждого класса по картам классификации**



In [ ]:
plot_dynamics(dw_areas_ha, DW_NAMES, DW_COLORS,
              'Динамика площадей Dynamic World', 'dynamic_world_dynamics.png')


### **7.3. Рассчитать все статистические показатели по картам классификации**



In [ ]:
dw_statistics = calculate_statistics(dw_areas_ha)
display(dw_statistics.round(2))
dw_statistics.to_csv(RESULTS / 'dynamic_world_statistics.csv', encoding='utf-8-sig')


### **7.4. Определить какие классы показали наибольшие изменения по картам классификации**



In [ ]:
significant_dw = ranked_changes(dw_statistics)


### **7.5. Создать сводную таблицу результатов по картам классификации**



In [ ]:
summary_dw = summary_table(dw_statistics)
display(summary_dw.round(2))
summary_dw.to_csv(RESULTS / 'dynamic_world_summary.csv', encoding='utf-8-sig')


## **8. Сравнение результатов полученных по результатам сегментации и картам классификаций**

**Задание:** Сведите классы обоих методов к укрупнённым классам по таблице 10. Сравните площади укрупнённых классов по годам, рассчитайте долю совпадающих пикселей для каждого года и матрицу совпадений для 2024 года.

#### **Таблица 10. Соответствие классов сегментации и Dynamic World**

| **Укрупнённый класс** | **Метки сегментации** | **Классы Dynamic World** |
|---|---|---|
| Вода | 0 | 0 — Вода |
| Снег/лед | 1 | 8 — Снег и лед |
| Застройка | 2 | 6 — Застройка |
| Растительность | 3, 4, 5 | 1 — Деревья, 2 — Трава, 3 — Затопл. растительность, 4 — Культуры, 5 — Кустарники |
| Открытая почва | 6 | 7 — Голая земля |
| Прочее | 7 | — |

In [ ]:
COARSE_NAMES = ['Вода', 'Снег/лед', 'Застройка', 'Растительность', 'Открытая почва', 'Прочее']
COARSE_COLORS = ['#419bdf', '#ffffff', '#c4281b', '#397d49', '#a59b8f', '#808080']
SEG_TO_COARSE = np.array([0, 1, 2, 3, 3, 3, 4, 5], dtype='int16')
DW_TO_COARSE = np.array([0, 3, 3, 3, 3, 3, 2, 4, 1], dtype='int16')


def remap_labels(labels, lookup):
    mapped = np.full(labels.shape, -1, dtype='int16')
    valid = (labels >= 0) & (labels < len(lookup))
    mapped[valid] = lookup[labels[valid]]
    return mapped


seg_coarse = {year: remap_labels(segmented[year], SEG_TO_COARSE) for year in YEARS}
dw_coarse = {year: remap_labels(dw_data[year], DW_TO_COARSE) for year in YEARS}
seg_coarse_area = count_pixels(seg_coarse, COARSE_NAMES, common_mask) * pixel_ha
dw_coarse_area = count_pixels(dw_coarse, COARSE_NAMES, common_mask) * pixel_ha
comparison_rows, agreement_rows = [], []
for year in YEARS:
    for name in COARSE_NAMES:
        comparison_rows.append({'Год': year, 'Класс': name,
                                'По индексам, га': seg_coarse_area.loc[name, year],
                                'Dynamic World, га': dw_coarse_area.loc[name, year],
                                'Разница, га': seg_coarse_area.loc[name, year] - dw_coarse_area.loc[name, year]})
    agreement_rows.append({'Год': year, 'Пиксели сравнения': int(common_mask.sum()),
                           'Совпадение, %': 100*np.mean(seg_coarse[year][common_mask] ==
                                                        dw_coarse[year][common_mask])})
comparison = pd.DataFrame(comparison_rows).set_index(['Год', 'Класс'])
agreement = pd.DataFrame(agreement_rows).set_index('Год')
display(comparison.round(2))
display(agreement.round(2))
comparison.to_csv(RESULTS / 'coarse_area_comparison.csv', encoding='utf-8-sig')
agreement.to_csv(RESULTS / 'agreement_by_year.csv', encoding='utf-8-sig')
plot_dynamics(seg_coarse_area, COARSE_NAMES, COARSE_COLORS,
              'Сравнение методов по укрупнённым классам', 'methods_comparison.png', dw_coarse_area)

confusion_ha = transition_matrix(seg_coarse[2024], dw_coarse[2024], common_mask, COARSE_NAMES)
confusion_pixels = (confusion_ha / pixel_ha).round().astype('int64')
confusion_pixels.index.name = 'Индексы / Dynamic World'
print('Матрица совпадений 2024 года: строки — индексы, столбцы — Dynamic World, пиксели.')
display(confusion_pixels)
confusion_pixels.to_csv(RESULTS / 'confusion_2024_pixels.csv', encoding='utf-8-sig')
confusion_ha.to_csv(RESULTS / 'confusion_2024_ha.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(9, 8), layout='constrained')
im = ax.imshow(confusion_ha, cmap='Blues')
ax.set_xticks(range(6), COARSE_NAMES, rotation=45, ha='right')
ax.set_yticks(range(6), COARSE_NAMES)
ax.set_xlabel('Dynamic World')
ax.set_ylabel('По спектральным индексам')
ax.set_title('Матрица совпадений, 2024')
for i in range(6):
    for j in range(6):
        ax.text(j, i, f'{confusion_ha.iloc[i,j]:.1f}', ha='center', va='center',
                 color='white' if confusion_ha.iloc[i,j] > confusion_ha.to_numpy().max()/2 else '#222222', fontsize=9)
fig.colorbar(im, ax=ax, label='Площадь, га', shrink=0.8)
fig.savefig(FIGURES / 'confusion_2024.png', dpi=120)
plt.show()


## **9. Выводы**

В выводах обязательно отразить:
1. **Количественная оценка динамики** (с конкретными цифрами для каждого класса)
2. **Выявленные тренды** (растущие, убывающие, стабильные классы)
3. **Скорость изменений** (га/год и %/год для ключевых классов)
4. **Пространственные закономерности** (где происходят основные изменения)
5. **Согласованность методов** (доля совпадающих пикселей по укрупнённым классам в % и основные причины расхождений)

In [ ]:
def number(value, signed=False):
    if not np.isfinite(value):
        return 'не определено'
    return f'{value:+.2f}' if signed else f'{value:.2f}'


lines = ['1. **Площади по индексам за 2019–2024 гг.:**']
for name, row in statistics.iterrows():
    lines.append(f"   - {name}: {row['S_2019, га']:.2f} → {row['S_2024, га']:.2f} га; "
                 f"изменение {row['ΔS, га']:+.2f} га ({number(row['Δ, %'], True)}%).")

def trend_list(summary, symbol):
    names = summary.index[summary['Тренд'] == symbol].tolist()
    return ', '.join(names) if names else 'нет'

lines += ['', '2. **Тренды по критериям задания.** '
          f"Рост: {trend_list(summary_seg, '↑')}; снижение: {trend_list(summary_seg, '↓')}; "
          f"малые колебания: {trend_list(summary_seg, '→')}; "
          f"выраженные колебания без устойчивого тренда: {trend_list(summary_seg, 'колебания')}."]
lines += ['', '3. **Скорость изменений ключевых классов:**']
for name in ['Вода', 'Застройка', 'Растительность', 'Открытая почва']:
    start, end = seg_coarse_area.loc[name, 2019], seg_coarse_area.loc[name, 2024]
    annual = (end - start) / 5
    rate = ((end/start)**(1/5)-1)*100 if start > 0 else np.nan
    lines.append(f'   - {name}: {annual:+.2f} га/год; {number(rate, True)}%/год.')

if changed_area_ha > 0:
    location = spatial_changes['Изменилось, га'].idxmax()
    location_area = spatial_changes.loc[location, 'Изменилось, га']
    text = (f'4. **Расположение изменений.** Класс сменился на {changed_area_ha:.2f} га '
            f'({100*changed.sum()/common_mask.sum():.2f}% области анализа). '
            f'Наибольшая площадь изменений — {location.lower()} ROI ({location_area:.2f} га). ')
    if not top_transitions.empty:
        top = top_transitions.iloc[0]
        text += f"Основной переход: {top['Из класса']} → {top['В класс']} ({top['Площадь, га']:.2f} га)."
else:
    text = '4. **Расположение изменений.** На общей маске смена классов не обнаружена.'
lines += ['', text]
lines += ['', '5. **Согласованность методов.** '
          f"В 2024 году совпали {agreement.loc[2024, 'Совпадение, %']:.2f}% пикселей; "
          f"среднее за шесть лет — {agreement['Совпадение, %'].mean():.2f}%. "
          'Различия связаны с порогами индексов, смешением почвы и застройки, '
          'а также медианным композитом Sentinel-2 и модой Dynamic World. '
          'Dynamic World — второй метод, а не независимая проверка по наземным данным.']
lines += ['', f'Сравнение выполнено на общей области {common_area_ha/100:.2f} км² '
          f'({100*common_mask.sum()/roi_mask.sum():.2f}% ROI). '
          'Изменение класса не всегда означает реальное изменение земного покрова.']
conclusions = '\n'.join(lines)
display(Markdown(conclusions))
_ = (RESULTS / 'conclusions.md').write_text(conclusions, encoding='utf-8')
